# BehaviorGPT Showcase

**BehaviorGPT** is trained on long sequences of things people actually did (viewed this, added that, bought the other) and predicts what comes next. Same idea as a language model predicting the next word, except the sequence is actions instead of text.

This notebook walks through the model in eight steps. Steps 1 to 4 use a sample Retail catalog. Steps 5 to 8 embed your own catalog and are optional.

## Step 1. Setup

You need an API key, free: sign in with your email at <a href="https://portal.unboxai.com" target="_blank" rel="noopener">portal.unboxai.com</a>. Put it in a `.env` file at the repo root as `UNBOXAI_API_KEY` (see `.env.example`); without one, the notebook asks for it.

The first cell installs the client if it is missing or older than 0.6.2, for example on Colab. In a clone set up with `uv sync` it does nothing.

The client packages your event history into HTTP requests. `market` sets the locale and `default_catalog_id` picks the catalog every call uses unless told otherwise. Here we browse a sample e-commerce catalog as a user from the US.

In [13]:
import importlib.metadata
import importlib.util
import subprocess
import sys

try:
    version = importlib.metadata.version("behaviorgpt")
    up_to_date = tuple(map(int, version.split(".")[:3])) >= (0, 6, 2)  # umap(wait=)
except importlib.metadata.PackageNotFoundError:
    up_to_date = False

# A venv made by uv has no pip, so only call it when something is missing.
if not up_to_date or not all(importlib.util.find_spec(m) for m in ("dotenv", "pandas")):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "behaviorgpt>=0.6.2", "python-dotenv", "pandas"],
        check=True,
    )

In [14]:
import os
from getpass import getpass

from behaviorgpt import UnboxAIClient, Search, View, Order
from dotenv import load_dotenv

load_dotenv(override=True)  # UNBOXAI_API_KEY from .env
if not os.environ.get("UNBOXAI_API_KEY"):
    os.environ["UNBOXAI_API_KEY"] = getpass("UNBOXAI_API_KEY: ")

In [ ]:
import json
items = []
index = 0
with open("../examples/Grocery_and_Gourmet_Food.jsonl",'r') as f:
    for aitem in f.readlines():
        items.append(json.loads(aitem))
        index = index + 1
        if index > 20000:
            break

import pandas as pd
df_groceries = pd.concat(items, axis=1)
df_groceries.head()        

In [ ]:
import pandas as pd

# Fast, native pandas loading for JSONL files
df_groceries = pd.read_json("../examples/Grocery_and_Gourmet_Food.jsonl", lines=True)

# Preview the first few rows
df_groceries.head()

In [ ]:
import pandas as pd

chunk_size = 50000  # Adjust based on your RAM
chunks = []

for chunk in pd.read_json("../examples/Grocery_and_Gourmet_Food.jsonl", lines=True, chunksize=chunk_size):
    # Optional: process or filter each chunk here if needed
    chunks.append(chunk)
    break

# Combine all chunks back into a single DataFrame
df_groceries = pd.concat(chunks, ignore_index=True)
df_groceries.head()

In [ ]:
df_groceries['date'] = pd.to_datetime(df_groceries['timestamp'], unit='s').dt.date


In [ ]:
df_groceries.to_csv('../examples/groceries_small.csv', index=False)

In [8]:
import pandas as pd

chunk_size = 50000  # Adjust based on your RAM
chunks = []

for chunk in pd.read_json("../examples/meta_Grocery_and_Gourmet_Food.jsonl", lines=True, chunksize=chunk_size):
    # Optional: process or filter each chunk here if needed
    chunks.append(chunk)
    print('done')

# Combine all chunks back into a single DataFrame
df_meta = pd.concat(chunks, ignore_index=True)
df_meta.head()

done
done
done
done
done
done
done
done
done
done
done
done
done


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Grocery,Dark Roast Pure Coffee,4.7,9,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'YouCut_20220Cafe Du Monde Coffee C...,Luzianne,"[Grocery & Gourmet Food, Beverages, Coffee]","{'Brand': 'Luzianne', 'Item Form': 'Ground', '...",B00NE08WM6,NaN,NaN,NaN
1,Grocery,PICARAS Galletas Peruanas Bañadas en Chocolate...,4.5,12,[2 Bags of PICARAS Galletas Bañadas en Chocola...,[2 Bags of PICARAS Galletas Bañadas en Chocola...,15.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Winters,"[Grocery & Gourmet Food, Breads & Bakery, Cook...","{'Flavor': 'chocolate', 'Unit Count': '12.00 C...",B084Q13Q5Q,NaN,NaN,NaN
2,Grocery,Chipped Beef and Gravy By Patterson's - Great ...,3.2,5,[Six (6) 8-ounce cans of delicious chipped bee...,[Delicious corned beef gravy is ready when you...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Pattersons,"[Grocery & Gourmet Food, Pantry Staples, Canne...",{'Package Dimensions': '8.4 x 5.8 x 3.4 inches...,B00KBRUYVM,NaN,NaN,NaN
3,Grocery,Asher's Sugar Free Milk Chocolate Cordial Cher...,5.0,6,[],[],29.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'No Sugar Added Chocolate Truffles'...,Generic,"[Grocery & Gourmet Food, Snacks & Sweets, Choc...","{'Brand': 'Generic', 'Item Form': 'Ball', 'Cho...",B0BN4PW255,NaN,NaN,NaN
4,Grocery,Messmer Peppermint 25 bags (6er pack),3.5,5,[],[],29.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Messmer,[],"{'Brand': 'Messmer', 'Item Form': 'Teabags', '...",B06X9DC27H,NaN,NaN,NaN


In [11]:
from sklearn.preprocessing import LabelEncoder

# Initialize the LabelEncoder
le = LabelEncoder()

# Fit and transform the main_category column into a new catalog_id (numerical) column
df_meta['catalog_id'] = le.fit_transform(df_meta['main_category'])

# Preview the changes to verify
df_meta[['main_category', 'catalog_id']].head()

,main_category,catalog_id
0,Grocery,17
1,Grocery,17
2,Grocery,17
3,Grocery,17
4,Grocery,17


In [10]:
df_meta['main_category'].nunique()

35

In [12]:
df_meta.to_csv('../examples/meta_groceries.csv', index=False)

In [37]:
import pandas as pd

df_meta = pd.read_csv('../examples/meta_groceries.csv')

/var/folders/41/s6mywxnn4czff_zmqxpdxxsw0000gn/T/ipykernel_15911/730168886.py:3: DtypeWarning: Columns (6,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta = pd.read_csv('../examples/meta_groceries.csv')


In [1]:
import pandas as pd

df_groceries = pd.read_csv('../examples/groceries_small.csv')

In [2]:
prediction  = """
AF5S43OTTG4RBXNGMC5OMKNEYP5A     1  B000MBKJKS      1.351854
AF5S43OTTG4RBXNGMC5OMKNEYP5A     2  B005BPVA08      1.342355
AF5S43OTTG4RBXNGMC5OMKNEYP5A     3  B086WNH45L      1.341638
AF5S43OTTG4RBXNGMC5OMKNEYP5A     4  B09HN8827V      1.341249
AF5S43OTTG4RBXNGMC5OMKNEYP5A     5  B00JLFDERQ      1.339658
AF5S43OTTG4RBXNGMC5OMKNEYP5A     6  B00PBCK9W6      1.339513
AF5S43OTTG4RBXNGMC5OMKNEYP5A     7  B08KL5G2B1      1.339122
AF5S43OTTG4RBXNGMC5OMKNEYP5A     8  B00BYHV1RO      1.338994
AF5S43OTTG4RBXNGMC5OMKNEYP5A     9  B00FBGM80I      1.338674
AF5S43OTTG4RBXNGMC5OMKNEYP5A    10  B0068W0PC2      1.338532
"""

In [36]:
df_groceries[(df_groceries['user_id']=='AF5S43OTTG4RBXNGMC5OMKNEYP5A') & (df_groceries['date']>'2023-03-01')]

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,date
34534,5,Raisins,They work good thanks👍😁,[],B0922YR24Z,B0BQWLNTLN,AF5S43OTTG4RBXNGMC5OMKNEYP5A,2023-03-07 12:29:44.778,0,True,2023-03-07
34535,5,Raisins,Love them they work good thanks 👍😀,[],B086Y768T1,B086Y768T1,AF5S43OTTG4RBXNGMC5OMKNEYP5A,2023-03-07 12:29:04.600,0,True,2023-03-07


In [34]:
# Group by user_id and find the user with the most entries
top_user_id = df_seg['user_id'].value_counts().idxmax()
top_user_count = df_seg['user_id'].value_counts().max()

print(f"User ID with max count: {top_user_id} ({top_user_count} interactions)")

# To filter and view all rows belonging to this user:
top_user_df = df_seg[df_seg['user_id'] == top_user_id]
top_user_df.head()

User ID with max count: AF5S43OTTG4RBXNGMC5OMKNEYP5A (53 interactions)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,date
34536,5,Coffee,I like it tastes good thanks,[],B008VT2NMK,B0BK4HGYHR,AF5S43OTTG4RBXNGMC5OMKNEYP5A,2023-02-21 19:18:06.476,0,True,2023-02-21
34537,5,Coffee,I thought it was bigger but it’s ok thanks the...,[],B01ERBZC7S,B01ERBZC7S,AF5S43OTTG4RBXNGMC5OMKNEYP5A,2023-02-03 14:12:58.922,0,True,2023-02-03
34538,5,Popcorn,Popcorn Taste good and low-calorie thanks,[],B01KQLULJW,B0C7H9X4S3,AF5S43OTTG4RBXNGMC5OMKNEYP5A,2023-01-26 16:38:26.359,0,True,2023-01-26
34539,5,Crackers,Crackers Taste good thanks,[],B07TQ99WFC,B0BRC6FLL1,AF5S43OTTG4RBXNGMC5OMKNEYP5A,2023-01-26 16:37:48.498,0,True,2023-01-26
34540,5,Gum,Taste good thanks,[],B0711V757H,B0BKMHS1XK,AF5S43OTTG4RBXNGMC5OMKNEYP5A,2023-01-26 16:36:58.466,0,True,2023-01-26


In [ ]:

# To see the full dataframe rows belonging to this top group:
top_group_df = df_groceries[df_groceries['id'] == top_group_id]
top_group_df.head()

In [4]:
client = UnboxAIClient(market="us", default_catalog_id="amazon_sample")


In [5]:
client.catalogs.list_catalogs()

[CatalogSummary(catalog_id='cat_7a85d820e9a381d3b447c9ac', catalog_name='groceries_small', status='pending', owned=True),
 CatalogSummary(catalog_id='cat_c814198390fa20c7328f2d1c', catalog_name='sample_catalog', status='ready', owned=True),
 CatalogSummary(catalog_id='cat_55125d6efa83a21df5360256', catalog_name='amazon_sample', status='ready', owned=False),
 CatalogSummary(catalog_id='cat_ec84f9d1140e3e6300ad5a41', catalog_name='art_catalog', status='ready', owned=False),
 CatalogSummary(catalog_id='hm_fashion', catalog_name='hm_fashion', status='ready', owned=False),
 CatalogSummary(catalog_id='cat_77d554465cbded0dc963769f', catalog_name='PixelRec200K', status='ready', owned=False),
 CatalogSummary(catalog_id='retail_catalog_500k', catalog_name='retail_catalog_500k', status='ready', owned=False)]

In [ ]:
res = client.complete(
    history=[], 
    catalog_id="cat_c814198390fa20c7328f2d1c", 
    limit=5
)
res.to_pandas()

In [ ]:
df = res.to_pandas()
df.columns

In [ ]:
df['timestamp'].unique()

`score` is the model's confidence in each item. Higher means more likely to come next.

Search is only one of the things this call does. With a different history, the same call recommends, personalizes and follows up on a purchase.

## Step 3. One call, many features

`complete` predicts what comes next in a sequence of events, the way a language model completes a sentence. The same call covers every feature in a store. The history, what the person did, picks most of them. `predict` picks the question: by default the next item they look at, and with `predict="purchase"` what else goes in the same purchase.

- **Cold start:** empty history. No personalization signal at all.
- **Search:** the last event is a query.
- **More like this:** the last event is an item.
- **Personalized search:** an item, then a query.
- **After buying:** an item viewed, then ordered.
- **Bought together:** an item, with `predict="purchase"`.

The catalog is the model's vocabulary: the complete list of things it can return. Nothing outside it ever comes back. The ids below are known items in the sample catalog, a retail store.

In [ ]:
# Example known item ids from the catalog in use by default

NIKE_PANTS = "paB08NYK61PJ"

EXPENSIVE_SNEAKERS_1 = "paB09DBFS1Y5"
EXPENSIVE_SNEAKERS_2 = "paB0BS1YV27T"

CHEAP_SNEAKERS_1 = "paB0CNFSW297"
CHEAP_SNEAKERS_2 = "paB08P34GCHQ"

In [ ]:
scenarios = {
    "cold start":          [],
    "search":              [Search("shoes")],
    "more like this":      [View(NIKE_PANTS)],
    "personalized search": [View(NIKE_PANTS), Search("shoes")],
    "after buying":        [View(NIKE_PANTS), Order([NIKE_PANTS])],
}

import pandas as pd


def by_history(results):
    """One table, the top items grouped under the history that produced them."""
    return pd.concat({label: res.to_pandas()[["name", "price", "score"]] for label, res in results.items()})


results = {label: client.complete(history=history, limit=3) for label, history in scenarios.items()}
results["bought together"] = client.complete(history=[View(NIKE_PANTS)], predict="purchase", limit=3)
by_history(results)

## Step 4. Context changes the answer

The model reads the whole sequence, not just the last thing typed. Same final query, two different preceding events:

In [ ]:
res_one = client.complete(history=[Search("nike"), Search("shoes")])
res_two = client.complete(history=[Search("adidas"), Search("shoes")])

display(res_one.to_pandas()[["name", "price", "score"]], res_two.to_pandas()[["name", "price", "score"]])

In [ ]:
display(res_one.to_pandas().head(5))

In [ ]:
display(res_one.to_pandas().columns)

In [ ]:
res_one_df = res_one.to_pandas()
res_one_df['group'].unique()

The word "shoes" did not change. What came before it did.

The next cell pushes further: the same query after clicks on expensive versus cheap sneakers. Nothing about price is passed to the model.

In [ ]:
QUERY = "shoes"

results = {}
for label, history in {
    "no history": [],
    "expensive":  [View(EXPENSIVE_SNEAKERS_1), View(EXPENSIVE_SNEAKERS_2)],
    "budget":     [View(CHEAP_SNEAKERS_1), View(CHEAP_SNEAKERS_2)],
}.items():
    res = client.complete(history=[*history, Search(QUERY)], catalog_id="retail_catalog_500k", limit=5)
    results[f"{label} (avg ${res.mean_price:.0f})"] = res

by_history(results)

The model picked up price sensitivity from the clicks alone.

## Step 5. Bring your own catalog (optional)

The model can only return items from its vocabulary, so to use it on your own data you embed your catalog. It can be products, artworks, articles or anything else people browse; <a href="https://github.com/Unbox-AI/aic-artworks" target="_blank" rel="noopener">Curate my wall</a> does it for 19 000 artworks. The first cell below converts `examples/sample_catalog.csv`, 100 items from the sample Retail catalog, to a parquet, so the steps run as they are. To embed your own catalog instead, give `catalog_from_csv` a CSV with the columns in <a href="../docs/catalog-format.md" target="_blank" rel="noopener">docs/catalog-format.md</a>, or build the parquet another way and set `CATALOG_PATH` to it.

`embed` uploads the file and starts a job: images are fetched, items are embedded, results are indexed. With `wait=True` the call blocks until the job finishes. That typically takes a few minutes; the 100 sample items take a few seconds. Without it, the call returns immediately and queries return `status` until processing completes. The returned `catalog_id` is private to your API key. `catalog_name` is your file name.

In [ ]:
from pathlib import Path

import httpx
from behaviorgpt import catalog_from_csv

SAMPLE_CSV = Path("../examples/sample_catalog.csv")
if not SAMPLE_CSV.exists():  # outside a clone of the repo
    SAMPLE_CSV = Path("sample_catalog.csv")
    url = "https://raw.githubusercontent.com/Unbox-AI/behaviorgpt/main/examples/sample_catalog.csv"
    SAMPLE_CSV.write_bytes(httpx.get(url).raise_for_status().content)

CATALOG_PATH = catalog_from_csv(SAMPLE_CSV)
CATALOG_PATH

In [ ]:
# CATALOG_PATH = "path/to/your_catalog.parquet"  # to embed your own catalog instead

job = client.embed(CATALOG_PATH, wait=True, timeout=1800.0)
job.model_dump()

## Step 6. Inspect the space

Each item is now a vector. Similar items land close together, unrelated ones far apart, and search and recommendation become nearest-neighbor lookups. The plot below projects the space to two dimensions. The map is drawn within about a minute after the catalog is ready, so the cell waits up to three minutes for it.

In [ ]:
from IPython.display import HTML
HTML(client.umap(catalog_id=job.catalog_id, wait=180))

Check the space by finding items similar to a random one. To test a specific item, replace `pick.id` with its id as a string.

In [ ]:
pick = client.random_item(job.catalog_id)
print(pick.data["name"])

response = client.similar_items(pick.id, catalog_id=job.catalog_id)
response.to_pandas()[["name", "price", "score"]]

## Step 7. Query your catalog

Same calls as steps 2 and 3, now with `catalog_id=job.catalog_id`. Change the queries to match what your catalog holds.

In [ ]:
res = client.complete(history=[Search("headphones")], catalog_id=job.catalog_id, limit=5)
res.to_pandas()[["name", "price", "score"]]

In [ ]:
SOME_ITEM = res.items[0].data["id"]

scenarios = {
    "cold start":          [],
    "search":              [Search("kitchen")],
    "more like this":      [View(SOME_ITEM)],
    "personalized search": [View(SOME_ITEM), Search("kitchen")],
}

by_history(
    {label: client.complete(catalog_id=job.catalog_id, history=history, limit=3) for label, history in scenarios.items()}
)

## Step 8. Try it in the demo

Once the catalog is embedded, use it in the interactive demo with BehaviorGPT as the recommendation engine:

1. Open <a href="https://behaviorgpt.unboxai.com/" target="_blank" rel="noopener">behaviorgpt.unboxai.com</a>.
2. Select the **BehaviorGPT V4.0-12.5B** model.
3. In the catalog dropdown, choose **Bring your own catalog** and enter your API key.

Images must be publicly reachable for the demo to display them.